In [ ]:
import os
import sys
import torch

os.environ['no_proxy'] = 'localhost,127.0.0.1,0.0.0.0,::1'
os.environ['NO_PROXY'] = 'localhost,127.0.0.1,0.0.0.0,::1'

import gradio as gr
import pandas as pd
import time
import json
import re


class HistoryManager:
    def __init__(self, root_dir="outputs/history"):
        self.root_dir = root_dir
        os.makedirs(self.root_dir, exist_ok=True)
        self.history_file = os.path.join(self.root_dir, "evaluation_history.md")
        self.headers = ["Timestamp", "Watermark", "Dataset", "Metrics (JSON)", "Detail Path"]

    def save_entry(self, wm_name, ds_name, df_result, path_wm):
        if df_result is None or df_result.empty: return

        try:
            timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
            metrics_dict = df_result.iloc[0].to_dict()
            metrics_str = json.dumps(metrics_dict, ensure_ascii=False)
            metrics_str = metrics_str.replace("|", "¦").replace("\n", " ") 
            metrics_code = f"`{metrics_str}`" 

            row = [timestamp, wm_name, ds_name, metrics_code, str(path_wm).replace("|", "¦")]
            row_str = "| " + " | ".join(str(x) for x in row) + " |\n"

            file_exists = os.path.exists(self.history_file)
            with open(self.history_file, 'a', encoding='utf-8') as f:
                if not file_exists:
                    f.write(f"# 📜 Evaluation History Log\n\n")
                    f.write("| " + " | ".join(self.headers) + " |\n")
                    f.write("| " + " | ".join(["---"] * len(self.headers)) + " |\n")
                f.write(row_str)
            print(f"[History] Saved entry to {self.history_file}")
        except Exception as e:
            print(f"[History] Failed to save entry: {e}")

    def load_history(self):
        if not os.path.exists(self.history_file):
            return pd.DataFrame(columns=self.headers)
        
        try:
            data = []
            with open(self.history_file, 'r', encoding='utf-8') as f:
                lines = f.readlines()
            
            start_parsing = False
            for line in lines:
                line = line.strip()
                if not line or line.startswith("#"): continue
                if set(line.replace("|", "").replace(" ", "")) == {"-"}:
                    start_parsing = True
                    continue
                
                if start_parsing and line.startswith("|"):
                    cells = [c.strip() for c in line.split("|")[1:-1]]
                    if len(cells) == len(self.headers):
                        data.append(cells)
            
            df = pd.DataFrame(data, columns=self.headers)
            return df
        except Exception as e:
            print(f"[History] Load error: {e}")
            return pd.DataFrame(columns=self.headers)

    def modify_history(self, action, row_index):

        if not os.path.exists(self.history_file) or row_index is None:
            return "File not found or no row selected"

        try:
            with open(self.history_file, 'r', encoding='utf-8') as f:
                lines = f.readlines()

            header_end_idx = -1
            for i, line in enumerate(lines):
                if set(line.strip().replace("|", "").replace(" ", "")) == {"-"}:
                    header_end_idx = i
                    break
            
            if header_end_idx == -1: return "Invalid Markdown format"
            
            data_start_idx = header_end_idx + 1
            data_indices = [i for i in range(data_start_idx, len(lines)) if lines[i].strip().startswith("|")]
            
            if row_index < 0 or row_index >= len(data_indices):
                return "Index out of bounds"
            
            abs_idx = data_indices[row_index]


            if action == 'delete':
                del lines[abs_idx]
            
            elif action == 'up':
                if row_index > 0: 
                    prev_abs_idx = data_indices[row_index - 1]
                    lines[abs_idx], lines[prev_abs_idx] = lines[prev_abs_idx], lines[abs_idx]
            
            elif action == 'down':
                if row_index < len(data_indices) - 1: 
                    next_abs_idx = data_indices[row_index + 1]
                    lines[abs_idx], lines[next_abs_idx] = lines[next_abs_idx], lines[abs_idx]

     
            with open(self.history_file, 'w', encoding='utf-8') as f:
                f.writelines(lines)
            
            return f"Action '{action}' success"

        except Exception as e:
            return f"Modify error: {e}"

history_manager = HistoryManager()

def export_results_to_markdown(df):
    if not isinstance(df, pd.DataFrame) or df.empty: return gr.update(visible=False), df
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    filename = f"eval_results_{timestamp}.md"
    try:
        markdown_table = df.to_markdown(index=False)
    except:
        return gr.update(visible=False), df
    try:
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(markdown_table)
    except: return gr.update(visible=False), df
    return gr.update(value=filename, visible=True), df

try:
    from src.gradio_adapter.quality_adapter import QualityGradioAdapter
except ImportError as e:
    print(f"\n❌ Import Error: {e}")
    sys.exit(1)

def list_files(subdir):
    path = os.path.join("configs", subdir)
    if not os.path.exists(path): return []
    return [f for f in os.listdir(path) if f.endswith('.yaml')]

exp_files = [f for f in list_files("experiments") if f.startswith('eval')]

GLOBAL_LISTS = {
    "wm": list_files("methods/watermarks"),
    "ds": list_files("datasets"),
    "exp": exp_files,
    "model": list_files("models"),
    "metrics": list_files("metrics")
}

backend = QualityGradioAdapter()

target_exp = "eval_quality_DwtDctSvd.yaml"
default_exp_name = target_exp if target_exp in GLOBAL_LISTS['exp'] else (GLOBAL_LISTS['exp'][0] if GLOBAL_LISTS['exp'] else None)

init_wm, init_ds, init_model, init_out, init_seed, init_max, init_bs = (
    "DwtDctSvd.yaml", "coco.yaml", "sd_v1_5.yaml",
    "outputs/quality_eval/DwtDctSvd_results", 42, 10, 1
)
if default_exp_name:
    try:
        params = backend.load_template_params(default_exp_name)
        if params:
            init_wm, init_ds, init_model, init_out, init_seed, init_max, init_bs = params
    except: pass


def tab1_logic(wm, model, prompt, seed):
    return backend.run_single_preview(wm, model, prompt, seed)

def tab2_logic(wm, ds, model, out_root, seed, max_samples, batch_size):
    full_log = ""
    for log_entry in backend.run_batch_generation(wm, ds, model, out_root, seed, max_samples, batch_size):
        full_log += str(log_entry)
        yield full_log

def tab3_logic(path_clean, path_wm, path_gt, path_meta, metrics, ds_name, wm_name): 
    full_log = ""
    current_df = pd.DataFrame()
    yield "Initializing...", pd.DataFrame()
    for log_entry, df in backend.run_batch_evaluation_custom(path_clean, path_wm, path_gt, path_meta, metrics, ds_name):
        if log_entry: full_log += str(log_entry)
        if isinstance(df, pd.DataFrame) and not df.empty: current_df = df.round(4)
        yield full_log, current_df
    
    if not current_df.empty:
        try:
            full_log += "\n💾 Saving to History (Markdown)..."
            history_manager.save_entry(wm_name, ds_name, current_df, path_wm)
            full_log += " Done.\n"
            yield full_log, current_df
        except: yield full_log, current_df

def load_template_ui(exp_name):
    results = backend.load_template_params(exp_name)
    if not results or len(results) < 7: return [gr.update()] * 9 
    return [
        gr.update(value=os.path.basename(str(results[0]))),
        gr.update(value=os.path.basename(str(results[1]))),
        gr.update(value=os.path.basename(str(results[2]))),
        gr.update(value=results[3]), gr.update(value=results[4]),
        gr.update(value=results[5]), gr.update(value=results[6]),
        gr.update(value=os.path.basename(str(results[0]))),
        gr.update(value=os.path.basename(str(results[1])))
    ]

def update_paths_ui(wm_name, ds_name, base_root_path):
    if not wm_name or not ds_name: return [gr.update()] * 4
    wm_clean = wm_name.replace('.yaml', '')
    ds_clean = ds_name.replace('.yaml', '')
    exp_base_path = os.path.join("outputs/quality_eval", f"{wm_clean}_results", f"{wm_clean}_{ds_clean}")
    return (
        os.path.join(exp_base_path, "clean"),
        os.path.join(exp_base_path, "watermarked"),
        os.path.join(exp_base_path, "ground_truth"),
        os.path.join(exp_base_path, "meta.json")
    )

def on_hist_select(evt: gr.SelectData):
    return evt.index[0]

def hist_action(action, row_idx):
    if row_idx is None:
        return gr.update(), "⚠️ Please select a row first."
    
    msg = history_manager.modify_history(action, int(row_idx))
    new_df = history_manager.load_history()
    return new_df, f"Action: {action} -> {msg}"




css = """
.gradio-container { background-color: #f0f2f5; font-family: 'Inter', sans-serif; }
.header-container { text-align: center; background: linear-gradient(to right, #f8f9fa, #e9ecef); padding: 25px; border-bottom: 3px solid #3182ce; margin-bottom: 20px; border-radius: 0 0 15px 15px; }
.header-title { font-size: 30px; font-weight: 800; color: #1a202c; }
.section-card { background: white; border-radius: 12px; padding: 20px; border: 1px solid #e5e7eb; box-shadow: 0 1px 3px rgba(0,0,0,0.1); margin-bottom: 15px; }
.section-header { font-size: 1.1rem; font-weight: 600; color: #111827; margin-bottom: 12px; border-bottom: 2px solid #f3f4f6; padding-bottom: 8px; }
.btn-done { background: linear-gradient(45deg, #11998e, #38ef7d) !important; color: white !important; font-weight: bold !important; }
.btn-gray { background: #e5e7eb !important; color: #374151 !important; font-weight: 600 !important; }
.code-box { font-family: monospace; font-size: 0.9em; }
"""

with gr.Blocks(title="Quality Pipeline", css=css, theme=gr.themes.Soft()) as app:
    
    gr.HTML("""
    <div class="header-container">
        <div class="header-title">🎨 AI Quality Analysis Studio</div>
        <div style="margin-top:5px; color:#4a5568;">Visual Fidelity & Metric Benchmarking</div>
    </div>
    """)

    state_hist_idx = gr.State(None)

    with gr.Tabs():
        
        with gr.Tab("🧪 Single Image Lab"):
            with gr.Row():
                with gr.Column(scale=1, elem_classes="section-card"):
                    gr.Markdown("<div class='section-header'>⚙️ Config</div>")
                    t1_wm = gr.Dropdown(GLOBAL_LISTS['wm'], label="Watermark Method", value="DwtDct.yaml")
                    t1_model = gr.Dropdown(GLOBAL_LISTS['model'], label="Model Config", value="sd_v1_5.yaml")
                    t1_prompt = gr.Textbox(label="Prompt", value="young, curly haired, redhead Natalie Portman", lines=3)
                    t1_seed = gr.Number(label="Seed", value=0, precision=0)
                    t1_btn = gr.Button("🚀 Generate Preview", elem_classes="btn-gray")
                    gr.Markdown("<div class='section-header'>📊 Instant Metrics</div>")
                    t1_df = gr.Dataframe(label="Quality Scores", interactive=False)
                    t1_msg = gr.Textbox(label="System Status", interactive=False)
                with gr.Column(scale=3):
                    with gr.Row():
                        t1_c = gr.Image(label="Clean", type="pil", height=350)
                        t1_w = gr.Image(label="Watermarked", type="pil", height=350)
                        t1_r = gr.Image(label="Residual (Gray x10)", type="pil", height=350)

        
        with gr.Tab("💾 Batch Generation"):
            with gr.Row():
                with gr.Column(scale=1, elem_classes="section-card"):
                    gr.Markdown("<div class='section-header'>📥 Batch Config</div>")
                    t2_exp = gr.Dropdown(GLOBAL_LISTS['exp'], label="📂 Load Experiment Template", value=default_exp_name)
                    gr.Markdown("---")
                    t2_wm = gr.Dropdown(GLOBAL_LISTS['wm'], label="Watermark Method", value=init_wm)
                    t2_ds = gr.Dropdown(GLOBAL_LISTS['ds'], label="Dataset", value=init_ds)
                    t2_model = gr.Dropdown(GLOBAL_LISTS['model'], label="Model Config", value=init_model)
                    t2_out = gr.Textbox(label="Output Path", value=init_out)
                    with gr.Row():
                        t2_seed = gr.Number(label="Start Seed", value=init_seed, precision=0)
                        t2_max = gr.Number(label="Max Samples", value=init_max, precision=0)
                        t2_bs = gr.Number(label="Batch Size", value=1, precision=0, minimum=1, maximum=16)
                    t2_btn = gr.Button("⚡ Run Generation", elem_classes="btn-done")
                with gr.Column(scale=2, elem_classes="section-card"):
                    gr.Markdown("<div class='section-header'>💻 Execution Log</div>")
                    t2_log = gr.TextArea(label="Terminal Output", lines=20, autoscroll=True, elem_classes="code-box")

        
        with gr.Tab("📈 Batch Evaluation"):
            with gr.Row():
                with gr.Column(scale=1, elem_classes="section-card"):
                    gr.Markdown("<div class='section-header'>📂 Evaluation Context</div>")
                    t3_wm = gr.Dropdown(GLOBAL_LISTS['wm'], label="Select Watermark", value=init_wm)
                    t3_ds = gr.Dropdown(GLOBAL_LISTS['ds'], label="Select Dataset", value=init_ds)
                    t3_root_base = gr.Textbox(label="Base Root", value="outputs/quality_eval", visible=False)
                    gr.Markdown("---")
                    gr.Markdown("**Target Paths (Auto-Updated)**")
                    with gr.Accordion("Path Details", open=True):
                        t3_p_clean = gr.Textbox(label="Clean Images Path")
                        t3_p_wm = gr.Textbox(label="Watermarked Images Path")
                        t3_p_gt = gr.Textbox(label="Ground Truth Path (Optional)")
                        t3_p_meta = gr.Textbox(label="Metadata Path (JSON)", value="meta.json")
                    gr.Markdown("<div class='section-header'>📐 Metrics Selection</div>")
                    t3_metrics = gr.CheckboxGroup(GLOBAL_LISTS['metrics'], label="Select Metrics", value=["basic_group.yaml"])
                    t3_btn = gr.Button("📊 Calculate & Save", elem_classes="btn-done")
                with gr.Column(scale=1, elem_classes="section-card"):
                    gr.Markdown("<div class='section-header'>💻 Progress Log</div>")
                    t3_log = gr.TextArea(label="Terminal Output", lines=22, autoscroll=True, elem_classes="code-box")
            with gr.Row():
                with gr.Column(elem_classes="section-card"):
                    gr.Markdown("<div class='section-header'>🏆 Aggregated Results</div>")
                    t3_res = gr.Dataframe(label="Metrics Table", interactive=False, wrap=True, type="pandas")
                    gr.Markdown("---")
                    with gr.Row():
                        t3_btn_export = gr.Button("💾 Export to Markdown (File)", elem_classes="btn-gray", scale=0)
                        t3_download = gr.File(label="Download File", visible=False, scale=1)

        
        with gr.Tab("📝 History"):
            with gr.Column(elem_classes="section-card"):
                gr.Markdown("<div class='section-header'>📜 Evaluation History (Markdown Storage)</div>")
                

                with gr.Row():
                    hist_refresh_btn = gr.Button("🔄 Refresh", elem_classes="btn-gray")
                    hist_up_btn = gr.Button("⬆️ Move Up", elem_classes="btn-gray")
                    hist_down_btn = gr.Button("⬇️ Move Down", elem_classes="btn-gray")
                    hist_del_btn = gr.Button("🗑️ Delete Row", variant="stop")
                
                hist_msg = gr.Textbox(label="Status", lines=1, interactive=False)


                init_history_df = history_manager.load_history()
                
                hist_table = gr.Dataframe(
                    value=init_history_df, 
                    headers=history_manager.headers,
                    label="Records (Click a row to select)",
                    interactive=False,
                    wrap=True,
                    datatype=["str", "str", "str", "markdown", "str"]
                )

    
    t1_btn.click(tab1_logic, [t1_wm, t1_model, t1_prompt, t1_seed], [t1_c, t1_w, t1_r, t1_df, t1_msg])
    
    t2_exp.change(load_template_ui, inputs=[t2_exp], outputs=[t2_wm, t2_ds, t2_model, t2_out, t2_seed, t2_max, t2_bs, t3_wm, t3_ds])
    t2_btn.click(tab2_logic, [t2_wm, t2_ds, t2_model, t2_out, t2_seed, t2_max, t2_bs], [t2_log])
    
    t3_wm.change(update_paths_ui, inputs=[t3_wm, t3_ds, t3_root_base], outputs=[t3_p_clean, t3_p_wm, t3_p_gt, t3_p_meta])
    t3_ds.change(update_paths_ui, inputs=[t3_wm, t3_ds, t3_root_base], outputs=[t3_p_clean, t3_p_wm, t3_p_gt, t3_p_meta])
    t3_btn.click(tab3_logic, inputs=[t3_p_clean, t3_p_wm, t3_p_gt, t3_p_meta, t3_metrics, t3_ds, t3_wm], outputs=[t3_log, t3_res])
    t3_btn_export.click(export_results_to_markdown, inputs=[t3_res], outputs=[t3_download, t3_res])

    

    hist_refresh_btn.click(history_manager.load_history, inputs=[], outputs=[hist_table])
    

    hist_table.select(on_hist_select, inputs=[], outputs=[state_hist_idx])
    

    hist_up_btn.click(lambda idx: hist_action("up", idx), inputs=[state_hist_idx], outputs=[hist_table, hist_msg])
    hist_down_btn.click(lambda idx: hist_action("down", idx), inputs=[state_hist_idx], outputs=[hist_table, hist_msg])
    hist_del_btn.click(lambda idx: hist_action("delete", idx), inputs=[state_hist_idx], outputs=[hist_table, hist_msg])


    app.load(update_paths_ui, inputs=[t3_wm, t3_ds, t3_root_base], outputs=[t3_p_clean, t3_p_wm, t3_p_gt, t3_p_meta])

if __name__ == "__main__":
    app.queue().launch(share=True)